In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

image_name = "9.png" #replace it with the image name
image = Image.open(image_name). convert("RGB")
texture = np.array(image)

seed_size = 3
output_size = 64

output = np.zeros((output_size, output_size, 3), dtype=np.uint8) #0-255
filled_mask = np.zeros((output_size, output_size), dtype=bool) #check which pixels in the generated image have been filled

height, width, _ = texture.shape
i = np.random.randint(0, height - seed_size)
j = np.random.randint(0, width - seed_size)
seed_patch = texture[i:i+seed_size, j:j+seed_size, :]

new_center = output_size // 2
seed_center = seed_size // 2
output[new_center - seed_center:new_center + seed_center + 1, new_center - seed_center:new_center + seed_center + 1, :] = seed_patch
filled_mask[new_center - seed_center:new_center + seed_center + 1, new_center - seed_center:new_center + seed_center + 1] = True

plt.imshow(output)
plt.show()

def frontier(filled_mask, window_size):
  frontier = []
  half = window_size // 2
  h, w = filled_mask.shape

  for y in range(h): #(y, x) is the center coordinate of the window, go through all possible window positions and check if there's a neighbor in the window
    for x in range(w):
      if filled_mask[y, x]:
        continue

      y_start = max(0, y - half)
      y_end = min(h, y + half + 1)
      x_start = max(0, x - half)
      x_end = min(w, x + half + 1)

      window = filled_mask[y_start:y_end, x_start:x_end]
      if np.any(window): #at least one filled neighbor
        frontier.append((y, x))

  return frontier

def gaussian_weight(window_size):
  half = window_size // 2
  y, x = np.mgrid[-half:half + 1, -half:half + 1] #two 2D arrays for row indices and column indices to determine the distance with the center
  sigma = window_size / 6.4
  g = np.exp(-(x**2 + y**2) / (2 * sigma**2))
  return g / g.sum()

def matching(texture, output, filled_mask, y, x, window_size, gaussian_mask):
  height, width, _ = texture.shape
  half = window_size // 2
  minimum_error = float('inf')
  best_patches = []

  y_start = max(0, y - half)
  y_end = min(output.shape[0], y + half + 1)
  x_start = max(0, x - half)
  x_end = min(output.shape[1], x + half + 1)

  output_window = output[y_start:y_end, x_start:x_end, :]
  mask_window = filled_mask[y_start:y_end, x_start:x_end]
  gw_y_start = half - (y - y_start)
  gw_y_end = gw_y_start + (y_end - y_start)
  gw_x_start = half - (x - x_start)
  gw_x_end = gw_x_start + (x_end - x_start)
  gaussian_crop = gaussian_mask[gw_y_start:gw_y_end, gw_x_start:gw_x_end]

  if not np.any(mask_window):
      return None

  for a in range(half, height - half):
    for b in range(half, width - half):
      candidate = texture[a - (y - y_start):a + (y_end - y), b - (x - x_start):b + (x_end - x), :]
      if output_window.shape[:2] != gaussian_crop.shape or output_window.shape != candidate.shape:
        continue

      differnce = (output_window - candidate) * mask_window[..., None]
      ssd = np.sum(gaussian_crop[..., None] * (differnce**2))

      if ssd < minimum_error:
        minimum_error = ssd
        best_patches = [(a, b)]
      elif ssd <= minimum_error * 1.1:
        best_patches.append((a, b))

  if not best_patches:
    return None

  best_a, best_b = best_patches[np.random.randint(len(best_patches))]
  center_pixel = texture[best_a, best_b]

  return center_pixel

window_size = 3
total_pixels = output_size * output_size

# Precompute the Gaussian mask once
g_mask = gaussian_weight(window_size)

# Main synthesis loop
num_filled = np.sum(filled_mask)

while num_filled < total_pixels:
    print(f"Filled: {num_filled}/{total_pixels}")

    frontier_pixels = frontier(filled_mask, window_size)
    print(f"Frontier size: {len(frontier_pixels)}")
    np.random.shuffle(frontier_pixels)

    working = False  # to track whether we made any changes this round

    for y, x in frontier_pixels:
        pixel = matching(texture, output, filled_mask, y, x, window_size, g_mask)
        if pixel is None:
          continue

        output[y, x] = pixel
        filled_mask[y, x] = True
        num_filled += 1
        working = True
        break

    if not working:
        print("try different parameters")
        break

# Show final result
plt.imshow(output)
plt.title("Final Synthesized Texture")
plt.axis('off')
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: '9.png'